### 2序列模型
#### 2.1计算题
给定一个字符序列"ababc"，假设采用一阶马尔可夫模型（即 $p(x_{t} | x_{t-1})$），使用拉普拉斯平滑（加1平滑）估计以下条件概率：
1. $p('a' | 'b')$
2. $p('c' | 'b')$

（词汇表为 $\{'a', 'b', 'c'\}$，计算时考虑所有可能转移，包括未出现的情况。）

#### 解答
##### 步骤1：统计转移计数
序列 `ababc` 共包含4个相邻字符对：`a→b`、`b→a`、`a→b`、`b→c`。
统计所有转移的计数如下表：

| 前驱字符 | 转移到a的次数 | 转移到b的次数 | 转移到c的次数 | 前驱总出现次数 |
|----------|---------------|---------------|---------------|----------------|
| a        | 0             | 2             | 0             | 2              |
| b        | 1             | 0             | 1             | 2              |
| c        | 0             | 0             | 0             | 0              |

词汇表大小 $V=3$。

##### 步骤2：拉普拉斯平滑公式
加1平滑的条件概率计算公式为：
$$
p(x_t | x_{t-1}) = \frac{\text{count}(x_{t-1} \to x_t) + 1}{\text{count}(x_{t-1}) + V}
$$

##### 步骤3：计算目标概率
1. 计算 $p('a' | 'b')$：
$$
p(a|b) = \frac{\text{count}(b\to a) + 1}{\text{count}(b) + V} = \frac{1+1}{2+3} = \frac{2}{5} = 0.4
$$

2. 计算 $p('c' | 'b')$：
$$
p(c|b) = \frac{\text{count}(b\to c) + 1}{\text{count}(b) + V} = \frac{1+1}{2+3} = \frac{2}{5} = 0.4
$$

In [1]:
import re
from collections import Counter

def preprocess_text(text, n):
    # 步骤1：转换为小写，去除标点，仅保留字母和空格
    text_lower = text.lower()
    # 正则匹配所有非小写字母和非空格的字符，替换为空
    text_clean = re.sub(r'[^a-z ]', '', text_lower)
    
    # 步骤2：按空格分词，自动过滤连续空格产生的空字符串
    words = [word for word in text_clean.split() if word]
    
    # 步骤3：构建按词频降序排序的词汇表，同频按字母序保证确定性，ID从0开始
    word_counts = Counter(words)
    # 排序规则：优先词频降序，次选字母升序
    sorted_words = sorted(word_counts.items(), key=lambda x: (-x[1], x[0]))
    vocab = {word: idx for idx, (word, _) in enumerate(sorted_words)}
    
    # 步骤4：滑动窗口生成特征与标签，无后续词则忽略
    features = []
    labels = []
    # 遍历所有存在下一个词的窗口，保证每个特征都有对应标签
    for i in range(len(words) - n):
        # 生成长度为n的特征序列
        features.append(words[i : i + n])
        # 对应下一个词作为标签
        labels.append(words[i + n])
    
    return vocab, (features, labels)

# 测试示例
if __name__ == "__main__":
    test_text = "The time machine"
    vocab, (feats, labels) = preprocess_text(test_text, n=2)
    print("词汇表:", vocab)
    print("特征序列:", feats)
    print("标签列表:", labels)

词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征序列: [['the', 'time']]
标签列表: ['machine']


### 3 循环神经网络
#### 3.1 理论计算题
考虑一个线性RNN（无偏置），定义为 $h_{t}=W_{hh} h_{t-1}+W_{hx} x_{t}$，输出 $o_{t}=W_{oh} h_{t}$。假设损失函数为平方损失 $L=\frac{1}{2} \sum_{t=1}^{T}(o_{t}-y_{t})^{2}$。推导损失对权重 $W_{hh}$ 的梯度表达式（通过时间反向传播，展开到所有时间步），并说明梯度消失或爆炸的条件。

#### 解答
##### 符号定义
- 隐藏状态维度为 $H$，输入维度为 $D$，输出维度为 $O$
- 权重形状：$W_{hh} \in \mathbb{R}^{H \times H}$，$W_{hx} \in \mathbb{R}^{H \times D}$，$W_{oh} \in \mathbb{R}^{O \times H}$
- 第$t$步单步损失：$L_t = \frac{1}{2}(o_t - y_t)^2$，总损失 $L = \sum_{t=1}^T L_t$

##### 梯度推导（通过时间反向传播 BPTT）
根据链式法则，总梯度为各时间步损失梯度的累加：
$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \frac{\partial L_t}{\partial W_{hh}}$$

#### 步骤1：单步损失对输出、隐藏状态的梯度
单步损失对输出的梯度：
$$\frac{\partial L_t}{\partial o_t} = o_t - y_t$$

单步损失对当前隐藏状态的梯度：
$$\delta_t = \frac{\partial L_t}{\partial h_t} = \left(\frac{\partial o_t}{\partial h_t}\right)^\top \frac{\partial L_t}{\partial o_t} = W_{oh}^\top (o_t - y_t)$$

#### 步骤2：时间维度的梯度反向传播
隐藏状态存在时间依赖：$h_t$ 依赖 $h_{t-1}$，因此第$t$步的损失会反向传播到所有更早的隐藏状态。
对任意 $k \leq t$，第$t$步损失对第$k$步隐藏状态的梯度：
$$\frac{\partial L_t}{\partial h_k} = \left( \frac{\partial h_{k+1}}{\partial h_k} \cdot \frac{\partial h_{k+2}}{\partial h_{k+1}} \cdots \frac{\partial h_t}{\partial h_{t-1}} \right)^\top \cdot \delta_t$$
由隐藏状态更新公式得 $\frac{\partial h_i}{\partial h_{i-1}} = W_{hh}$，代入后：
$$\frac{\partial L_t}{\partial h_k} = \left(W_{hh}^\top\right)^{t-k} \cdot \delta_t$$

#### 步骤3：隐藏状态对权重的梯度
第$k$步隐藏状态对 $W_{hh}$ 的梯度：
$$\frac{\partial h_k}{\partial W_{hh}} = h_{k-1}^\top$$

#### 步骤4：总梯度表达式
结合链式法则，将所有时间步的梯度累加，得到总梯度：
$$
\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \sum_{k=1}^t \left(W_{hh}^\top\right)^{t-k} \delta_t \cdot h_{k-1}^\top
$$

工程上常用递推形式实现（从最后一步反向遍历）：
$$
\begin{cases}
dh_t = \delta_t + W_{hh}^\top \cdot dh_{t+1} \quad (dh_{T+1} = 0) \\
dW_{hh} = \sum_{t=1}^T dh_t \cdot h_{t-1}^\top
\end{cases}
$$

##### 梯度消失与爆炸的条件
梯度在时间反向传播过程中，需要连续乘以 $W_{hh}^\top$，其数值稳定性由 $W_{hh}$ 的**谱半径**$\rho$（矩阵最大特征值的绝对值）决定：
1. **梯度消失**：当 $\boldsymbol{\rho < 1}$ 时，长距离梯度会随时间步指数级衰减至0，模型无法学习长距离依赖关系；
2. **梯度爆炸**：当 $\boldsymbol{\rho > 1}$ 时，梯度会随时间步指数级增长至无穷，导致训练数值不稳定、模型发散。

In [2]:
import numpy as np

def rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN单步前向传播（带tanh激活）
    参数:
        x_t: (batch_size, input_size) 当前时间步输入
        h_prev: (batch_size, hidden_size) 上一时间步隐藏状态
        W_hx: (input_size, hidden_size) 输入权重矩阵
        W_hh: (hidden_size, hidden_size) 隐藏状态权重矩阵
        b_h: (hidden_size,) 偏置项
    返回:
        h_t: (batch_size, hidden_size) 当前时间步隐藏状态
        cache: 反向传播所需的缓存变量
    """
    # 计算线性变换部分
    z_t = np.dot(x_t, W_hx) + np.dot(h_prev, W_hh) + b_h
    # 应用tanh激活函数得到当前隐藏状态
    h_t = np.tanh(z_t)
    
    # 缓存反向传播需要的变量
    cache = (x_t, h_prev, h_t, W_hx, W_hh)
    return h_t, cache

def rnn_cell_backward(dh_next, cache):
    """
    RNN单步反向传播（计算梯度，不更新参数）
    参数:
        dh_next: (batch_size, hidden_size) 损失对当前隐藏状态h_t的上游梯度
    返回:
        dx_t: (batch_size, input_size) 损失对输入x_t的梯度
        dh_prev: (batch_size, hidden_size) 损失对上一隐藏状态h_prev的梯度
        dW_hx: (input_size, hidden_size) 损失对输入权重W_hx的梯度
        dW_hh: (hidden_size, hidden_size) 损失对隐藏权重W_hh的梯度
        db_h: (hidden_size,) 损失对偏置b_h的梯度
    """
    x_t, h_prev, h_t, W_hx, W_hh = cache
    
    # 步骤1：计算tanh激活的梯度：d(tanh(z))/dz = 1 - tanh(z)^2 = 1 - h_t^2
    dz_t = dh_next * (1 - h_t ** 2)
    
    # 步骤2：计算各参数和变量的梯度
    db_h = np.sum(dz_t, axis=0)    # 偏置梯度：batch维度求和
    dW_hx = np.dot(x_t.T, dz_t)    # 输入权重梯度
    dx_t = np.dot(dz_t, W_hx.T)    # 输入梯度
    dW_hh = np.dot(h_prev.T, dz_t) # 隐藏权重梯度
    dh_prev = np.dot(dz_t, W_hh.T) # 上一隐藏状态梯度
    
    return dx_t, dh_prev, dW_hx, dW_hh, db_h

# 测试验证
if __name__ == "__main__":
    batch_size = 2
    input_size = 3
    hidden_size = 4
    
    x_t = np.random.randn(batch_size, input_size)
    h_prev = np.random.randn(batch_size, hidden_size)
    W_hx = np.random.randn(input_size, hidden_size)
    W_hh = np.random.randn(hidden_size, hidden_size)
    b_h = np.random.randn(hidden_size)
    
    h_t, cache = rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h)
    dh_next = np.random.randn(batch_size, hidden_size)
    dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_cell_backward(dh_next, cache)
    
    print("当前隐藏状态形状:", h_t.shape)
    print("dW_hh形状:", dW_hh.shape)
    print("dh_prev形状:", dh_prev.shape)

当前隐藏状态形状: (2, 4)
dW_hh形状: (4, 4)
dh_prev形状: (2, 4)


### 4 高级循环神经网络
#### 4.1 理论计算题
假设一个深度双向RNN，有L层，每层隐藏单元数为H，输入维度为D，输出维度为O（仅考虑最后输出层）。计算该模型的参数总数（包括所有全连接层的权重和偏置），忽略嵌入层和输出层之前的投影，明确给出表达式。

#### 解答
深度双向RNN的每一层都包含**前向、后向两个独立的RNN单元**，每个方向的输出在时间步维度拼接后，作为下一层的输入。

##### 逐层参数计算
#### 1. 第1层（输入层）
输入维度为 $D$，单个方向的RNN单元包含3类参数：
- 输入权重：$D \times H$
- 隐藏状态权重：$H \times H$
- 偏置：$H$

双向共2个方向，因此第1层总参数：
$$
2 \times (D \cdot H + H^2 + H) = 2H(D + H + 1)
$$

#### 2. 第2层 ~ 第L层（中间层）
上一层双向输出拼接后维度为 $2H$，作为当前层的输入。单个方向RNN的输入权重维度变为 $2H \times H$，其余参数与第1层一致。
单个方向参数：$2H \cdot H + H^2 + H = 3H^2 + H$
双向单层总参数：$2 \times (3H^2 + H) = 6H^2 + 2H$
共 $L-1$ 层中间层，总参数：
$$
(L-1) \cdot (6H^2 + 2H) = 2(L-1)H(3H + 1)
$$

#### 3. 最终输出全连接层
输入为最后一层双向拼接的隐藏状态，维度 $2H$，输出维度 $O$，包含权重和偏置：
$$
2H \cdot O + O = O(2H + 1)
$$

#### 总参数表达式
将三部分相加，得到模型总参数数：
$$
\begin{align*}
\text{总参数} &= 2H(D + H + 1) + 2(L-1)H(3H + 1) + O(2H + 1) \\
&= 2HD + (6L - 4)H^2 + 2LH + 2HO + O
\end{align*}
$$

In [3]:
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        # 初始化双向单层RNN，适配输入格式 (seq_len, batch, input_dim)
        self.bi_rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            bidirectional=True,
            batch_first=False
        )
    
    def forward(self, X):
        """
        双向RNN前向传播
        参数:
            X: (seq_len, batch, input_dim) 输入序列
        返回:
            step_outputs: (seq_len, batch, 2*hidden_dim) 每个时间步双向拼接的隐藏状态
            final_repr: (batch, 2*hidden_dim) 最终时间步的拼接隐藏状态（序列全局表示）
        """
        # 前向计算：outputs是每个时间步的双向拼接结果
        # h_n形状: (num_directions, batch, hidden_dim)
        step_outputs, h_n = self.bi_rnn(X)
        
        # 拼接前向和后向的最终隐藏状态，得到全局序列表示
        final_repr = torch.cat([h_n[0], h_n[1]], dim=-1)
        
        return step_outputs, final_repr

# 测试验证
if __name__ == "__main__":
    seq_len = 5
    batch = 3
    input_dim = 4
    hidden_dim = 6
    
    X = torch.randn(seq_len, batch, input_dim)
    encoder = BiRNNEncoder(input_dim, hidden_dim)
    step_outputs, final_repr = encoder(X)
    
    print("逐时间步输出形状:", step_outputs.shape)
    print("最终序列表示形状:", final_repr.shape)

逐时间步输出形状: torch.Size([5, 3, 12])
最终序列表示形状: torch.Size([3, 12])


#### 5 嵌入向量
#### 5.1 理论计算题
在Skip-gram模型中，给定中心词 $w_c$ 和上下文词 $w_o$，使用负采样（采样 K 个负样本）。推导其损失函数（对数似然）的表达式，并说明如何从噪声分布中采样负样本。假设词向量为 $v_c$，$u_o$，负样本词向量为 $u_{n_k}$，写出完整的目标函数。

#### 解答
##### 损失函数推导
Skip-gram负采样的核心思想是：将“预测上下文词”转化为“区分正样本和负样本”的二分类任务，通过最大化正样本的概率、最小化负样本的概率来学习词向量。

#### 1. 概率定义
使用sigmoid函数将词向量的内积转化为概率：
- 正样本（真实上下文词）概率：$P(w_o | w_c) = \sigma(v_c^\top u_o) = \frac{1}{1 + e^{-v_c^\top u_o}}$
- 负样本（非上下文词）概率：对于第$k$个负样本 $n_k$，其“不是上下文词”的概率为 $P(\neg w_{n_k} | w_c) = \sigma(-v_c^\top u_{n_k})$

#### 2. 单个样本的对数似然
对于一个中心词-正上下文词对，以及采样得到的$K$个负样本，对数似然为正样本对数概率与所有负样本对数概率之和：
$$
\log P(w_o | w_c) = \log \sigma(v_c^\top u_o) + \sum_{k=1}^K \log \sigma\left(-v_c^\top u_{n_k}\right)
$$

#### 3. 完整目标函数
模型训练目标是最大化所有中心词-上下文对的对数似然，因此损失函数（负对数似然）为：
$$
J = -\sum_{(c,o) \in \mathcal{D}} \left[ \log \sigma(v_c^\top u_o) + \sum_{k=1}^K \log \sigma\left(-v_c^\top u_{n_k}\right) \right]
$$
其中 $\mathcal{D}$ 为语料中所有中心词-上下文词对的集合。

#### 负样本采样方法
负样本从**噪声分布**中采样，实践中普遍使用**一元语法的3/4次方分布**，公式为：
$$
P_n(w) = \frac{\text{count}(w)^{3/4}}{\sum_{w' \in V} \text{count}(w')^{3/4}}
$$
其中 $\text{count}(w)$ 是词$w$在语料中的出现次数。

该分布的作用是：平滑原始词频的差异，适当降低高频词的采样概率、提升低频词的采样概率，在训练效率和效果之间取得最优平衡。采样时需保证负样本不等于当前的正上下文词。

In [4]:
import torch
import torch.nn.functional as F

def cbow_forward_loss(context_indices, target_indices, W, W_out):
    """
    CBOW模型前向传播 + 完整softmax交叉熵损失计算
    参数:
        context_indices: (batch_size, context_size) 每个样本的上下文词索引
        target_indices: (batch_size,) 每个样本对应的中心词标签（真实值）
        W: (V, d) 输入嵌入权重矩阵，V为词汇表大小，d为嵌入维度
        W_out: (d, V) 输出权重矩阵
    返回:
        loss: 批次平均交叉熵损失
    """
    batch_size, context_size = context_indices.shape
    
    # 步骤1：根据索引取出上下文词的嵌入向量
    context_embeddings = W[context_indices]  # 形状: (batch_size, context_size, d)
    
    # 步骤2：对上下文向量取平均，得到隐藏层表示
    hidden = torch.mean(context_embeddings, dim=1)  # 形状: (batch_size, d)
    
    # 步骤3：通过输出权重计算词汇表上的得分
    scores = torch.matmul(hidden, W_out)  # 形状: (batch_size, V)
    
    # 步骤4：计算完整softmax的交叉熵损失
    log_probs = F.log_softmax(scores, dim=-1)
    loss = F.nll_loss(log_probs, target_indices)
    
    return loss

# 测试验证
if __name__ == "__main__":
    V = 10      # 词汇表大小
    d = 4       # 嵌入维度
    batch_size = 2
    context_size = 2
    
    context_indices = torch.tensor([[1, 3], [0, 2]])
    target_indices = torch.tensor([2, 1])
    W = torch.randn(V, d, requires_grad=True)
    W_out = torch.randn(d, V, requires_grad=True)
    
    loss = cbow_forward_loss(context_indices, target_indices, W, W_out)
    print("交叉熵损失值:", loss.item())

交叉熵损失值: 3.0273313522338867


### 6 注意力机制
### 6.1 理论计算题
给定查询矩阵 $Q \in \mathbb{R}^{2 ×4}$，键矩阵 $K \in \mathbb{R}^{3 ×4}$，值矩阵 $V \in \mathbb{R}^{3 ×5}$。计算缩放点积注意力（无掩码）的输出矩阵，要求写出中间步骤（先计算得分矩阵，再softmax，再加权求和）。使用 $score =Q K^{T} / \sqrt{d_{k}}(d_{k}=4)$。可以只列出数值计算过程（用符号或具体数值）。

#### 解答
缩放点积注意力的通用公式为：
$$\text{Attention}(Q,K,V) = \text{softmax}\left( \frac{Q K^\top}{\sqrt{d_k}} \right) V$$

以下通过具体数值演示完整计算过程：
##### 设定数值
设查询、键、值矩阵分别为：
$$
Q = \begin{bmatrix} 1 & 0 & 2 & 0 \\ 0 & 1 & 0 & 1 \end{bmatrix}_{2\times4}, \quad
K = \begin{bmatrix} 1 & 1 & 0 & 0 \\ 0 & 1 & 1 & 0 \\ 0 & 0 & 1 & 1 \end{bmatrix}_{3\times4}, \quad
V = \begin{bmatrix} 1 & 2 & 3 & 4 & 5 \\ 6 & 7 & 8 & 9 & 10 \\ 11 & 12 & 13 & 14 & 15 \end{bmatrix}_{3\times5}
$$
已知 $d_k=4$，因此缩放因子 $\sqrt{d_k}=2$。

##### 步骤1：计算得分矩阵 $S = \frac{Q K^\top}{\sqrt{d_k}}$
先计算 $Q K^\top$：
$$
Q K^\top = \begin{bmatrix} 1\times1+0\times1+2\times0+0\times0 & 1\times0+0\times1+2\times1+0\times0 & 1\times0+0\times0+2\times1+0\times1 \\ 0\times1+1\times1+0\times0+1\times0 & 0\times0+1\times1+0\times1+1\times0 & 0\times0+1\times0+0\times1+1\times1 \end{bmatrix}
= \begin{bmatrix} 1 & 2 & 2 \\ 1 & 1 & 1 \end{bmatrix}
$$
除以缩放因子2，得到得分矩阵：
$$
S = \frac{1}{2} \begin{bmatrix} 1 & 2 & 2 \\ 1 & 1 & 1 \end{bmatrix} = \begin{bmatrix} 0.5 & 1 & 1 \\ 0.5 & 0.5 & 0.5 \end{bmatrix}
$$

##### 步骤2：对得分矩阵做行方向softmax，得到注意力权重
softmax公式：$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$

- 第1行权重：
  分母 $e^{0.5} + e^1 + e^1 \approx 1.6487 + 2.7183 + 2.7183 = 7.0853$
  $$
  A_1 = \left[ \frac{e^{0.5}}{7.0853},\ \frac{e^1}{7.0853},\ \frac{e^1}{7.0853} \right] \approx [0.2327,\ 0.3836,\ 0.3836]
  $$

- 第2行权重：三个得分相同，权重均匀分布
  $$
  A_2 = \left[ \frac{1}{3},\ \frac{1}{3},\ \frac{1}{3} \right] \approx [0.3333,\ 0.3333,\ 0.3333]
  $$

最终注意力权重矩阵：
$$
A = \begin{bmatrix} 0.2327 & 0.3836 & 0.3836 \\ 0.3333 & 0.3333 & 0.3333 \end{bmatrix}
$$

##### 步骤3：加权求和得到输出矩阵 $O = A V$
$$
\begin{align*}
O &= \begin{bmatrix} 0.2327 & 0.3836 & 0.3836 \\ 0.3333 & 0.3333 & 0.3333 \end{bmatrix} \begin{bmatrix} 1 & 2 & 3 & 4 & 5 \\ 6 & 7 & 8 & 9 & 10 \\ 11 & 12 & 13 & 14 & 15 \end{bmatrix} \\
&\approx \begin{bmatrix} 6.771 & 7.771 & 8.771 & 9.771 & 10.771 \\ 6 & 7 & 8 & 9 & 10 \end{bmatrix}_{2\times5}
\end{align*}
$$

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=4, num_heads=2):
        super().__init__()
        # 校验d_model可以被头数整除
        assert d_model % num_heads == 0, "d_model必须能被num_heads整除"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 每个头的维度
        
        # Q、K、V的线性投影层
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        # 多头拼接后的最终线性层
        self.W_o = nn.Linear(d_model, d_model)
    
    def split_heads(self, x):
        """
        将最后一维拆分为num_heads个头，调整维度顺序便于注意力计算
        输入: (seq_len, batch, d_model)
        输出: (num_heads, batch, seq_len, d_k)
        """
        seq_len, batch, _ = x.shape
        x = x.view(seq_len, batch, self.num_heads, self.d_k)
        return x.permute(2, 1, 0, 3)
    
    def forward(self, X):
        """
        多头注意力前向传播
        参数:
            X: (seq_len, batch, d_model) 输入序列
        返回:
            output: (seq_len, batch, d_model) 多头注意力输出，形状与输入一致
        """
        seq_len, batch, _ = X.shape
        
        # 步骤1：线性投影得到Q、K、V
        Q = self.W_q(X)
        K = self.W_k(X)
        V = self.W_v(X)
        
        # 步骤2：拆分为多个头
        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)
        
        # 步骤3：对每个头并行计算缩放点积注意力
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        attn_weights = F.softmax(scores, dim=-1)
        attn_output = torch.matmul(attn_weights, V)
        
        # 步骤4：拼接所有头的输出
        attn_output = attn_output.permute(2, 1, 0, 3).contiguous()
        attn_output = attn_output.view(seq_len, batch, self.d_model)
        
        # 步骤5：经过最终线性层
        output = self.W_o(attn_output)
        
        return output

# 测试验证
if __name__ == "__main__":
    d_model = 4
    num_heads = 2
    seq_len = 3
    batch = 2
    
    X = torch.randn(seq_len, batch, d_model)
    mha = MultiHeadAttention(d_model, num_heads)
    output = mha(X)
    
    print("输入形状:", X.shape)
    print("输出形状:", output.shape)

输入形状: torch.Size([3, 2, 4])
输出形状: torch.Size([3, 2, 4])
